In [1]:
!pip -q install pandas requests tqdm tenacity rapidfuzz

import pandas as pd, requests, time, unicodedata, re, math
from tqdm import tqdm
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from rapidfuzz import fuzz


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 45.7 MB/s eta 0:00:00


In [2]:
INPUT_XLSX   = "pesquisadores.xlsx"   # seu arquivo
SHEET_NAME   = 0
COL_NOME     = "Pesquisador_nome"
COL_INST     = "inst_norm"            # ajuda a desambiguar
OUTPUT_XLSX  = "pesquisadores_metrics_openalex.xlsx"

# Opções
COMPUTAR_I10 = True      # se ficar lento, troque para False
REQUESTS_POR_SEG = 2     # OpenAlex tolera bem; reduza se necessário
EMAIL_CONTATO   = "mailto:seu_email@exemplo.com"  # ponha seu email (boas práticas da API)

BASE = "https://api.openalex.org"
HEADERS = {"User-Agent": f"OpenAlex for research ({EMAIL_CONTATO})"}
SLEEP = 1.0 / REQUESTS_POR_SEG


In [3]:
def norm(s:str) -> str:
    if not isinstance(s,str): return ""
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = s.lower().strip()
    s = re.sub(r"\s+", " ", s)
    return s

@retry(stop=stop_after_attempt(4), wait=wait_exponential(multiplier=1, min=1, max=20),
       retry=retry_if_exception_type(Exception))
def oa_get(url, params=None):
    r = requests.get(url, params=params or {}, headers=HEADERS, timeout=30)
    if r.status_code >= 400:
        raise RuntimeError(f"HTTP {r.status_code}: {r.text[:200]}")
    time.sleep(SLEEP)
    return r.json()

def buscar_autores(nome:str, inst:str=""):
    params = {
        "search": nome,        # busca por nome
        "per-page": 10,
        "mailto": EMAIL_CONTATO
    }
    data = oa_get(f"{BASE}/authors", params)
    resultados = data.get("results", [])
    # se não achou nada e tem inst, tenta com o termo da inst
    if not resultados and inst:
        params["search"] = f"{nome} {inst}"
        data = oa_get(f"{BASE}/authors", params)
        resultados = data.get("results", [])
    return resultados

def pontuar_candidato(c, nome_norm:str, inst_norm:str):
    # nome: fuzzy (parcial) + exato
    nome_cand = norm(c.get("display_name",""))
    score_nome = max(
        fuzz.token_set_ratio(nome_norm, nome_cand),
        fuzz.partial_ratio(nome_norm, nome_cand)
    )/100.0  # 0..1

    # instituição atual/conhecida
    aff = ""
    last_inst = (c.get("last_known_institution") or {}).get("display_name", "")
    if last_inst:
        aff = norm(last_inst)
    elif c.get("affiliations"):
        aff = norm(" ".join([a.get("institution",{}).get("display_name","") for a in c["affiliations"] if a.get("institution")]))

    score_inst = 0.0
    if inst_norm and aff:
        score_inst = 1.0 if inst_norm in aff or aff in inst_norm else fuzz.partial_ratio(inst_norm, aff)/100.0

    # bônus por consistência bibliométrica (autores “vazios” tendem a ser homônimos)
    works = c.get("works_count") or 0
    cited = c.get("cited_by_count") or 0
    score_meta = 0.0
    if works>=3 and cited>=5:
        score_meta = 0.1
    if works>=10 and cited>=50:
        score_meta = 0.2

    return 0.6*score_nome + 0.3*score_inst + 0.1*score_meta

def escolher_melhor(candidatos, nome:str, inst:str):
    nome_norm = norm(nome)
    inst_norm = norm(inst)
    if not candidatos: return None, 0.0
    scored = [(pontuar_candidato(c, nome_norm, inst_norm), c) for c in candidatos]
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[0][1], scored[0][0]

def coletar_autor_metrics(author_id:str, computar_i10=True):
    # /authors/{id} já traz h_index, works_count, cited_by_count
    a = oa_get(f"{BASE}/authors/{author_id}", params={"mailto": EMAIL_CONTATO})
    out = {
        "openalex_id": a.get("id"),
        "display_name": a.get("display_name"),
        "last_known_institution": (a.get("last_known_institution") or {}).get("display_name"),
        "h_index": a.get("h_index"),
        "works_count": a.get("works_count"),
        "cited_by_count": a.get("cited_by_count"),
    }
    if not computar_i10:
        out["i10_index"] = None
        return out

    # calcular i10 percorrendo works (só citações)
    i10 = 0
    cursor = "*"
    while True:
        params = {
            "filter": f"author.id:{author_id}",
            "select": "cited_by_count",
            "per-page": 200,
            "cursor": cursor,
            "mailto": EMAIL_CONTATO
        }
        w = oa_get(f"{BASE}/works", params)
        for it in w.get("results", []):
            if (it.get("cited_by_count") or 0) >= 10:
                i10 += 1
        cursor = w.get("meta", {}).get("next_cursor")
        if not cursor: break
        # limite de segurança (evitar percorrer autores com milhares de artigos)
        if i10 > 500: break
    out["i10_index"] = i10
    return out


In [4]:
df = pd.read_excel(INPUT_XLSX, sheet_name=SHEET_NAME)
assert COL_NOME in df.columns, f"Coluna '{COL_NOME}' não encontrada"
if COL_INST not in df.columns:
    df[COL_INST] = ""

# criar colunas de saída
for c in ["oa_display_name","oa_institution","oa_author_id",
          "h_index","works_count","cited_by_count","i10_index",
          "match_score","match_status","notes"]:
    if c not in df.columns:
        df[c] = None

for i, row in tqdm(df.iterrows(), total=len(df)):
    nome = str(row[COL_NOME]).strip()
    inst = str(row[COL_INST]).strip() if pd.notna(row[COL_INST]) else ""
    if not nome:
        df.at[i,"match_status"] = "empty_name"
        continue

    try:
        cand = buscar_autores(nome, inst)
        best, score = escolher_melhor(cand, nome, inst)
        if not best:
            df.at[i,"match_status"] = "no_match"
            continue

        # coletar métricas do melhor
        author_id = best.get("id")
        met = coletar_autor_metrics(author_id, computar_i10=COMPUTAR_I10)

        df.at[i,"oa_author_id"]   = met["openalex_id"]
        df.at[i,"oa_display_name"]= met["display_name"]
        df.at[i,"oa_institution"] = met["last_known_institution"]
        df.at[i,"h_index"]        = met["h_index"]
        df.at[i,"works_count"]    = met["works_count"]
        df.at[i,"cited_by_count"] = met["cited_by_count"]
        df.at[i,"i10_index"]      = met["i10_index"]
        df.at[i,"match_score"]    = round(score,3)
        df.at[i,"match_status"]   = "matched"
    except Exception as e:
        df.at[i,"match_status"] = "error"
        df.at[i,"notes"]        = str(e)

df.to_excel(OUTPUT_XLSX, index=False)
print("OK ->", OUTPUT_XLSX)

# Amostra rápida
display(df[[COL_NOME, COL_INST, "oa_display_name","oa_institution","h_index","i10_index","cited_by_count","works_count","match_status","match_score"]].head(15))


100%|██████████| 541/541 [33:05<00:00,  3.67s/it]

OK -> pesquisadores_metrics_openalex.xlsx


,Pesquisador_nome,inst_norm,oa_display_name,oa_institution,h_index,i10_index,cited_by_count,works_count,match_status,match_score
0,João Paulo Sinnecker,CENTRO BRASILEIRO DE PESQUISAS FISICAS,J.P. Sinnecker,None,None,49,1731,136,matched,0.82
1,Andre Massafferri Rodrigues,CENTRO BRASILEIRO DE PESQUISAS FISICAS,A. Massafferri,None,None,55,48281,1377,matched,0.848
2,Maria de Lourdes Aguiar Oliveira,CENTRO DE PESQUISA GONCALO MONIZ - FIOCRUZ,Maria de Lourdes Aguiar Oliveira,None,None,24,775,50,matched,0.763
3,Rodrigo Guerino Stabeli,CENTRO DE PESQUISA GONCALO MONIZ - FIOCRUZ,Rodrigo G. Stábeli,None,None,93,3976,146,matched,0.723
4,Talita Mazon,Centro de Tecnologia da Informação Renato Archer,Talita Mazon,None,None,17,679,62,matched,0.92
5,Ricardo Rodrigues de Araujo,CENTRO FEDERAL DE EDUCACAO TECNOLOGICA CELSO S...,Ricardo Rodrigues de Araujo,None,None,1,73,17,matched,0.833
6,Ronney Arismel Mancebo Boloy,CENTRO FEDERAL DE EDUCACAO TECNOLOGICA CELSO S...,Ronney Arismel Mancebo Boloy,None,None,13,536,52,matched,0.833
7,Pedro Manuel Calas Lopes Pacheco,CENTRO FEDERAL DE EDUCACAO TECNOLOGICA CELSO S...,Pedro Manuel Calas Lopes Pacheco,None,None,39,1543,104,matched,0.833
8,André Luiz Mota,CENTRO FEDERAL DE EDUCACAO TECNOLOGICA DE MINA...,André Mota,None,None,11,510,28,matched,0.798
9,Carlos Magno Martins Cosme,CENTRO FEDERAL DE EDUCACAO TECNOLOGICA DE MINA...,Carlos Magno Martins Cosme,None,None,0,13,6,matched,0.804


In [5]:
from google.colab import files
files.download("pesquisadores_metrics_openalex.xlsx")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
!pip -q install pandas requests tenacity

import pandas as pd, requests, time, unicodedata, re
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

INPUT = "pesquisadores_metrics_openalex.xlsx"
OUTPUT = "pesquisadores_metrics_openalex_FIX.xlsx"
EMAIL = "mailto:seu_email@exemplo.com"
BASE = "https://api.openalex.org"
HDRS = {"User-Agent": f"OpenAlex fix ({EMAIL})"}

@retry(stop=stop_after_attempt(4), wait=wait_exponential(multiplier=1, min=1, max=20),
       retry=retry_if_exception_type(Exception))
def oa_get(url, params=None):
    r = requests.get(url, params=params or {}, headers=HDRS, timeout=30)
    r.raise_for_status()
    time.sleep(0.3)
    return r.json()

def recompute_i10(author_id: str):
    i10 = 0
    cursor = "*"
    while True:
        w = oa_get(f"{BASE}/works", params={
            "filter": f"author.id:{author_id}",
            "select": "cited_by_count",
            "per-page": 200,
            "cursor": cursor,
            "mailto": EMAIL
        })
        for it in w.get("results", []):
            if (it.get("cited_by_count") or 0) >= 10:
                i10 += 1
        cursor = w.get("meta", {}).get("next_cursor")
        if not cursor: break
    return i10

df = pd.read_excel(INPUT)

# só corrige quem tem ID e está sem h_index
mask = df["oa_author_id"].notna() & df["oa_author_id"].astype(str).str.len().gt(0) & df["h_index"].isna()

fix_count = 0
for i, row in df[mask].iterrows():
    author_id = str(row["oa_author_id"])
    try:
        a = oa_get(f"{BASE}/authors/{author_id}", params={"mailto": EMAIL})
        df.at[i, "h_index"]        = a.get("h_index")
        df.at[i, "works_count"]    = a.get("works_count")
        df.at[i, "cited_by_count"] = a.get("cited_by_count")
        # opcional: recalc i10
        # df.at[i, "i10_index"]      = recompute_i10(author_id)
        fix_count += 1
    except Exception as e:
        df.at[i, "notes"] = f"fix_error: {e}"

df.to_excel(OUTPUT, index=False)
print("Corrigidos:", fix_count, "| Salvo em:", OUTPUT)


Corrigidos: 528 | Salvo em: pesquisadores_metrics_openalex_FIX.xlsx


In [9]:
from google.colab import files
files.download("pesquisadores_metrics_openalex_FIX.xlsx")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
!pip -q install pandas requests tenacity

import pandas as pd, requests, time
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

INPUT  = "pesquisadores_metrics_openalex_FIX.xlsx"   # << seu arquivo atual
OUTPUT = "pesquisadores_metrics_openalex_FIX2.xlsx"  # << arquivo corrigido
EMAIL  = "mailto:seu_email@exemplo.com"              # troque para seu e-mail
BASE   = "https://api.openalex.org"
HDRS   = {"User-Agent": f"OpenAlex fix2 ({EMAIL})"}
SLEEP  = 0.25                                        # 4 req/s aprox. (ajuste se necessário)

@retry(stop=stop_after_attempt(4), wait=wait_exponential(multiplier=1, min=1, max=20),
       retry=retry_if_exception_type(Exception))
def oa_get(url, params=None):
    r = requests.get(url, params=params or {}, headers=HDRS, timeout=30)
    r.raise_for_status()
    time.sleep(SLEEP)
    return r.json()

def compute_h_i10_from_works(author_id: str):
    """Percorre todas as obras do autor e calcula h-index e i10-index localmente."""
    cites = []
    cursor = "*"
    while True:
        data = oa_get(f"{BASE}/works", params={
            "filter": f"author.id:{author_id}",
            "select": "cited_by_count",
            "per-page": 200,
            "cursor": cursor,
            "mailto": EMAIL
        })
        for it in data.get("results", []):
            cites.append(it.get("cited_by_count") or 0)
        cursor = data.get("meta", {}).get("next_cursor")
        if not cursor:
            break
    cites.sort(reverse=True)
    h = 0
    for i, c in enumerate(cites, start=1):
        if c >= i: h = i
        else: break
    i10 = sum(1 for c in cites if c >= 10)
    return h, i10, len(cites), sum(cites)

df = pd.read_excel(INPUT)

# Garantir as colunas
for col in ["h_index","i10_index","works_count","cited_by_count","notes"]:
    if col not in df.columns:
        df[col] = None

mask_targets = df["oa_author_id"].notna() & df["oa_author_id"].astype(str).str.len().gt(0) & df["h_index"].isna()

fixed_via_summary = 0
fixed_via_local   = 0
errors            = 0

for i, row in df[mask_targets].iterrows():
    author_id = str(row["oa_author_id"])
    try:
        # 1) tenta preencher via resumo do autor
        a = oa_get(f"{BASE}/authors/{author_id}", params={"mailto": EMAIL})
        h = a.get("h_index")
        if h is not None:
            df.at[i, "h_index"]        = h
            df.at[i, "works_count"]    = a.get("works_count")
            df.at[i, "cited_by_count"] = a.get("cited_by_count")
            fixed_via_summary += 1
            continue  # já resolveu

        # 2) se o resumo não trouxe h_index, calcula localmente
        h_local, i10_local, works, cited = compute_h_i10_from_works(author_id)
        df.at[i, "h_index"]        = h_local
        df.at[i, "i10_index"]      = i10_local
        df.at[i, "works_count"]    = works
        df.at[i, "cited_by_count"] = cited
        fixed_via_local += 1

    except Exception as e:
        df.at[i, "notes"] = f"fix2_error: {e}"
        errors += 1

df.to_excel(OUTPUT, index=False)
print(f"Corrigidos via /authors: {fixed_via_summary} | Corrigidos via cálculo local: {fixed_via_local} | Erros: {errors}")
print("Salvo em:", OUTPUT)


Corrigidos via /authors: 0 | Corrigidos via cálculo local: 528 | Erros: 0
Salvo em: pesquisadores_metrics_openalex_FIX2.xlsx


In [11]:
from google.colab import files
files.download("pesquisadores_metrics_openalex_FIX2.xlsx")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
!pip -q install pandas requests tenacity

import pandas as pd, requests, time
from collections import Counter, defaultdict
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type


In [13]:
INPUT  = "pesquisadores_metrics_openalex_FIX2.xlsx"  # seu arquivo atual
OUTPUT = "pesquisadores_metrics_openalex_FIX2_AREAS.xlsx"
EMAIL  = "mailto:seu_email@exemplo.com"              # troque para seu e-mail (boa prática API)
BASE   = "https://api.openalex.org"
HDRS   = {"User-Agent": f"OpenAlex areas ({EMAIL})"}
SLEEP  = 0.25                                        # ~4 req/s


In [14]:
@retry(stop=stop_after_attempt(4), wait=wait_exponential(multiplier=1, min=1, max=20),
       retry=retry_if_exception_type(Exception))
def oa_get(url, params=None):
    r = requests.get(url, params=params or {}, headers=HDRS, timeout=30)
    r.raise_for_status()
    time.sleep(SLEEP)
    return r.json()

def pick_top(concepts, level_set, min_score=0.0):
    # concepts: lista de dicts com keys ['display_name', 'level', 'score']
    # retorna (nome, score) do melhor conceito no(s) nível(is) desejado(s)
    best = None
    for c in concepts:
        if c.get("level") in level_set and (c.get("score") or 0) >= min_score:
            if best is None or (c.get("score") or 0) > (best.get("score") or 0):
                best = c
    if not best:
        return None, None
    return best.get("display_name"), best.get("score")

def from_author_xconcepts(author_id):
    # tenta pegar diretamente x_concepts do autor
    a = oa_get(f"{BASE}/authors/{author_id}", params={"mailto": EMAIL, "select": "id,display_name,last_known_institution,x_concepts"})
    x = a.get("x_concepts") or []
    # normaliza campos essenciais
    concepts = [{"display_name": c.get("display_name"),
                 "level": c.get("level"),
                 "score": c.get("score")} for c in x if c]
    return concepts

def from_works_aggregate_concepts(author_id, max_pages=5):
    # agrega conceitos dos trabalhos (conceitos têm level e score por obra)
    agg = defaultdict(float)
    levels = {}
    cursor = "*"
    pages = 0
    while True and pages < max_pages:
        w = oa_get(f"{BASE}/works", params={
            "filter": f"author.id:{author_id}",
            "select": "id,concepts",
            "per-page": 200,
            "cursor": cursor,
            "mailto": EMAIL
        })
        for it in w.get("results", []):
            for c in it.get("concepts", []) or []:
                name = c.get("display_name")
                lvl  = c.get("level")
                sc   = c.get("score") or 0.0
                if name:
                    agg[(name, lvl)] += sc  # soma scores ao longo das obras
                    levels[name] = lvl
        cursor = w.get("meta", {}).get("next_cursor")
        pages += 1
        if not cursor:
            break
    # volta na forma de lista
    concepts = [{"display_name": n, "level": lvl, "score": s} for (n, lvl), s in agg.items()]
    return concepts


In [15]:
df = pd.read_excel(INPUT)
for c in ["grande_area_openalex","grande_area_score","area_openalex","area_score","areas_fonte"]:
    if c not in df.columns:
        df[c] = None

mask = df["oa_author_id"].notna() & df["oa_author_id"].astype(str).str.len().gt(0)

via_author = 0
via_works  = 0
for i, row in df[mask].iterrows():
    aid = str(row["oa_author_id"])
    concepts = []
    fonte = None
    try:
        # 1) tenta diretamente no autor
        concepts = from_author_xconcepts(aid)
        fonte = "author_x_concepts"
        # 2) fallback: agrega dos works se veio vazio
        if not concepts:
            concepts = from_works_aggregate_concepts(aid, max_pages=5)
            fonte = "works_concepts_agg"
    except Exception as e:
        df.at[i, "areas_fonte"] = f"erro: {e}"
        continue

    if fonte == "author_x_concepts":
        via_author += 1
    elif fonte == "works_concepts_agg":
        via_works += 1

    # Grande Área = melhor conceito level 0
    ga, ga_sc = pick_top(concepts, {0}, min_score=0.0)
    # Área = melhor conceito level 1–2
    ar, ar_sc = pick_top(concepts, {1,2}, min_score=0.0)

    df.at[i, "grande_area_openalex"] = ga
    df.at[i, "grande_area_score"]    = ga_sc
    df.at[i, "area_openalex"]        = ar
    df.at[i, "area_score"]           = ar_sc
    df.at[i, "areas_fonte"]          = fonte

df.to_excel(OUTPUT, index=False)
print(f"Conceitos via autor: {via_author} | via agregação de works: {via_works}")
print("Salvo em:", OUTPUT)


Conceitos via autor: 0 | via agregação de works: 0
Salvo em: pesquisadores_metrics_openalex_FIX2_AREAS.xlsx


In [16]:
!pip -q install pandas requests tenacity

import pandas as pd, requests, time
from collections import defaultdict, Counter
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

INPUT  = "pesquisadores_metrics_openalex_FIX2.xlsx"     # << seu arquivo atual
OUTPUT = "pesquisadores_metrics_openalex_FIX2_AREAS_FIX.xlsx"
EMAIL  = "mailto:seu_email@exemplo.com"                 # troque para seu e-mail (boa prática da API)
BASE   = "https://api.openalex.org"
HDRS   = {"User-Agent": f"OpenAlex areas-fix ({EMAIL})"}
SLEEP  = 0.25                                           # ~4 req/s (ajuste se necessário)

@retry(stop=stop_after_attempt(4), wait=wait_exponential(multiplier=1, min=1, max=20),
       retry=retry_if_exception_type(Exception))
def oa_get(url, params=None):
    r = requests.get(url, params=params or {}, headers=HDRS, timeout=30)
    r.raise_for_status()
    time.sleep(SLEEP)
    return r.json()

def aggregate_concepts_from_works(author_id: str, max_pages=None):
    """
    Varre TODAS as obras do autor na OpenAlex e agrega conceitos por:
      - freq[name, level]  (contagem de vezes em que aparece)
      - score_sum[name, level] (soma do 'score' ao longo das obras)
    max_pages=None -> sem limite (usa cursor até acabar).
    """
    freq = defaultdict(int)
    score_sum = defaultdict(float)
    cursor = "*"
    pages = 0

    while True:
        params = {
            "filter": f"author.id:{author_id}",
            "select": "id,concepts",
            "per-page": 200,
            "cursor": cursor,
            "mailto": EMAIL
        }
        data = oa_get(f"{BASE}/works", params)

        for work in data.get("results", []):
            for c in (work.get("concepts") or []):
                name  = c.get("display_name")
                level = c.get("level")
                sc    = c.get("score") or 0.0
                if name is None or level is None:
                    continue
                key = (name, level)
                freq[key] += 1
                score_sum[key] += sc

        cursor = data.get("meta", {}).get("next_cursor")
        pages += 1
        if not cursor:
            break
        if max_pages is not None and pages >= max_pages:
            break

    # transforma em lista de dicts
    concepts = []
    for (name, level), f in freq.items():
        concepts.append({
            "display_name": name,
            "level": level,
            "freq": f,
            "score_sum": score_sum[(name, level)]
        })
    return concepts

def pick_best(concepts, wanted_levels):
    """
    Escolhe o melhor conceito em 'wanted_levels':
      1º critério: maior freq
      2º critério: maior score_sum
    Retorna (name, freq, score_sum) ou (None, 0, 0.0)
    """
    pool = [c for c in concepts if c["level"] in wanted_levels]
    if not pool:
        return None, 0, 0.0
    pool.sort(key=lambda c: (c["freq"], c["score_sum"]), reverse=True)
    top = pool[0]
    return top["display_name"], top["freq"], top["score_sum"]

# --- Executa no arquivo
df = pd.read_excel(INPUT)

for c in ["grande_area_openalex","grande_area_freq","grande_area_score_sum",
          "area_openalex","area_freq","area_score_sum","areas_fonte"]:
    if c not in df.columns:
        df[c] = None

mask = df["oa_author_id"].notna() & df["oa_author_id"].astype(str).str.len().gt(0)

ok = 0
no_concepts = 0
errors = 0

for i, row in df[mask].iterrows():
    aid = str(row["oa_author_id"])
    try:
        concepts = aggregate_concepts_from_works(aid, max_pages=None)  # varre tudo
        if not concepts:
            df.at[i, "areas_fonte"] = "no_concepts"
            no_concepts += 1
            continue

        ga_name, ga_freq, ga_sc = pick_best(concepts, {0})
        ar_name, ar_freq, ar_sc = pick_best(concepts, {1,2})

        df.at[i, "grande_area_openalex"]   = ga_name
        df.at[i, "grande_area_freq"]       = ga_freq
        df.at[i, "grande_area_score_sum"]  = ga_sc
        df.at[i, "area_openalex"]          = ar_name
        df.at[i, "area_freq"]              = ar_freq
        df.at[i, "area_score_sum"]         = ar_sc
        df.at[i, "areas_fonte"]            = "works_full_scan"
        ok += 1

    except Exception as e:
        df.at[i, "areas_fonte"] = f"error: {e}"
        errors += 1

df.to_excel(OUTPUT, index=False)
print(f"OK: {ok} | Sem conceitos: {no_concepts} | Erros: {errors}")
print("Salvo em:", OUTPUT)


OK: 528 | Sem conceitos: 0 | Erros: 0
Salvo em: pesquisadores_metrics_openalex_FIX2_AREAS_FIX.xlsx
